# Market Making Strategy Example

This notebook demonstrates how to use market making strategies to provide liquidity and profit from bid-ask spreads.

## Setup

First, let's import the necessary modules and set up our environment.

In [ ]:
import os
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import asyncio

from src.strategies.market_maker import (
    MarketMaker,
    MarketMakingStyle,
    OrderBook,
    MarketState
)
from src.data.binance_fetcher import BinanceFetcher

## Configuration

Load and configure market making settings.

In [ ]:
# Load configuration
with open('configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Initialize market maker
market_maker = MarketMaker(
    config=config,
    exchange='binance',
    base_spread=0.001,
    min_spread=0.0005,
    max_spread=0.01,
    order_size=0.01,
    max_inventory=1.0
)

print("Market maker initialized with:")
print(f"Base spread: {market_maker.base_spread:.4f}")
print(f"Min spread: {market_maker.min_spread:.4f}")
print(f"Max spread: {market_maker.max_spread:.4f}")
print(f"Order size: {market_maker.order_size:.4f}")
print(f"Max inventory: {market_maker.max_inventory:.4f}")

## Market Analysis

Analyze market conditions and order book state.

In [ ]:
async def analyze_market():
    """Analyze market conditions."""
    symbol = 'BTC/USDT'
    order_book, market_state = await market_maker.fetch_market_state(symbol)
    
    print("\nOrder Book Analysis:")
    print(f"Mid Price: ${order_book.mid_price:,.2f}")
    print(f"Spread: {order_book.spread*100:.4f}%")
    print(f"Bid Depth: {order_book.depth['bid_depth']:.4f}")
    print(f"Ask Depth: {order_book.depth['ask_depth']:.4f}")
    
    print("\nMarket State:")
    print(f"Volatility: {market_state.volatility*100:.2f}%")
    print(f"Volume: {market_state.volume:.4f}")
    print(f"Trend: {market_state.trend:.4f}")
    print(f"Imbalance: {market_state.imbalance:.4f}")
    
    return order_book, market_state

# Analyze market
order_book, market_state = await analyze_market()

## Order Book Visualization

Visualize the order book state and market making opportunities.

In [ ]:
def visualize_orderbook(order_book: OrderBook):
    """Visualize order book state."""
    plt.figure(figsize=(15, 5))
    
    # Plot order book depth
    plt.subplot(1, 2, 1)
    bid_prices = list(order_book.bids.keys())
    bid_sizes = list(order_book.bids.values())
    ask_prices = list(order_book.asks.keys())
    ask_sizes = list(order_book.asks.values())
    
    plt.bar(bid_prices, bid_sizes, color='g', alpha=0.5, label='Bids')
    plt.bar(ask_prices, ask_sizes, color='r', alpha=0.5, label='Asks')
    plt.axvline(order_book.mid_price, color='k', linestyle='--')
    plt.title('Order Book Depth')
    plt.xlabel('Price')
    plt.ylabel('Size')
    plt.legend()
    
    # Plot cumulative depth
    plt.subplot(1, 2, 2)
    cum_bids = np.cumsum(bid_sizes)
    cum_asks = np.cumsum(ask_sizes)
    
    plt.plot(bid_prices, cum_bids, 'g-', label='Cumulative Bids')
    plt.plot(ask_prices, cum_asks, 'r-', label='Cumulative Asks')
    plt.axvline(order_book.mid_price, color='k', linestyle='--')
    plt.title('Cumulative Depth')
    plt.xlabel('Price')
    plt.ylabel('Cumulative Size')
    plt.legend()
    
    plt.tight_layout()
    plt.show()

# Visualize order book
visualize_orderbook(order_book)

## Spread Analysis

Analyze optimal spread calculation and price levels.

In [ ]:
def analyze_spreads(market_state: MarketState):
    """Analyze spread calculations."""
    # Calculate spreads for different volatilities
    volatilities = np.linspace(0.01, 0.05, 10)
    spreads = []
    
    for vol in volatilities:
        state = MarketState(
            price=market_state.price,
            volatility=vol,
            volume=market_state.volume,
            trend=market_state.trend,
            imbalance=market_state.imbalance,
            timestamp=market_state.timestamp
        )
        spreads.append(market_maker.calculate_optimal_spread(state))
    
    # Plot spread analysis
    plt.figure(figsize=(15, 5))
    
    # Spread vs Volatility
    plt.subplot(1, 2, 1)
    plt.plot(volatilities * 100, np.array(spreads) * 100)
    plt.title('Spread vs Volatility')
    plt.xlabel('Volatility (%)')
    plt.ylabel('Spread (%)')
    
    # Current spread levels
    plt.subplot(1, 2, 2)
    spread = market_maker.calculate_optimal_spread(market_state)
    bid_price, ask_price = market_maker.calculate_order_prices(market_state)
    
    plt.axvline(bid_price, color='g', label='Bid')
    plt.axvline(market_state.price, color='k', linestyle='--', label='Mid')
    plt.axvline(ask_price, color='r', label='Ask')
    plt.title('Price Levels')
    plt.xlabel('Price')
    plt.legend()
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nOptimal Spread: {spread*100:.4f}%")
    print(f"Bid Price: ${bid_price:,.2f}")
    print(f"Ask Price: ${ask_price:,.2f}")

# Analyze spreads
analyze_spreads(market_state)

## Order Size Analysis

Analyze order size calculations and inventory management.

In [ ]:
def analyze_order_sizes(market_state: MarketState):
    """Analyze order size calculations."""
    # Calculate sizes for different inventory levels
    inventory_levels = np.linspace(-1.0, 1.0, 21)
    bid_sizes = []
    ask_sizes = []
    
    for inv in inventory_levels:
        market_maker.inventory = inv
        bid_size, ask_size = market_maker.calculate_order_sizes(market_state)
        bid_sizes.append(bid_size)
        ask_sizes.append(ask_size)
    
    # Plot size analysis
    plt.figure(figsize=(15, 5))
    
    # Order sizes vs Inventory
    plt.subplot(1, 2, 1)
    plt.plot(inventory_levels, bid_sizes, 'g-', label='Bid Size')
    plt.plot(inventory_levels, ask_sizes, 'r-', label='Ask Size')
    plt.axvline(0, color='k', linestyle='--')
    plt.title('Order Sizes vs Inventory')
    plt.xlabel('Inventory')
    plt.ylabel('Order Size')
    plt.legend()
    
    # Current order sizes
    plt.subplot(1, 2, 2)
    bid_size, ask_size = market_maker.calculate_order_sizes(market_state)
    plt.bar(['Bid', 'Ask'], [bid_size, ask_size],
            color=['g', 'r'], alpha=0.5)
    plt.title('Current Order Sizes')
    plt.ylabel('Size')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nBid Size: {bid_size:.4f}")
    print(f"Ask Size: {ask_size:.4f}")

# Analyze order sizes
analyze_order_sizes(market_state)

## Market Making Simulation

Simulate market making strategy with paper trading.

In [ ]:
async def simulate_market_making(duration: int = 60):
    """Simulate market making strategy."""
    start_time = time.time()
    simulation_data = []
    
    while time.time() - start_time < duration:
        # Fetch market state
        order_book, market_state = await market_maker.fetch_market_state(
            'BTC/USDT'
        )
        
        # Calculate orders
        bid_price, ask_price = market_maker.calculate_order_prices(
            market_state
        )
        bid_size, ask_size = market_maker.calculate_order_sizes(
            market_state
        )
        
        # Place orders
        success = await market_maker.place_orders(
            'BTC/USDT',
            bid_price,
            ask_price,
            bid_size,
            ask_size
        )
        
        # Record data
        simulation_data.append({
            'timestamp': datetime.now(),
            'mid_price': market_state.price,
            'spread': order_book.spread,
            'inventory': market_maker.inventory,
            'position_value': market_maker.position_value,
            'realized_pnl': market_maker.realized_pnl,
            'unrealized_pnl': market_maker.unrealized_pnl
        })
        
        # Update plot
        if len(simulation_data) % 5 == 0:
            df = pd.DataFrame(simulation_data)
            
            plt.clf()
            plt.figure(figsize=(15, 10))
            
            # Price and Spread
            plt.subplot(2, 2, 1)
            plt.plot(df['timestamp'], df['mid_price'])
            plt.title('Mid Price')
            plt.xticks(rotation=45)
            
            # Inventory
            plt.subplot(2, 2, 2)
            plt.plot(df['timestamp'], df['inventory'])
            plt.axhline(0, color='k', linestyle='--')
            plt.title('Inventory')
            plt.xticks(rotation=45)
            
            # P&L
            plt.subplot(2, 2, 3)
            plt.plot(df['timestamp'], df['realized_pnl'],
                    label='Realized')
            plt.plot(df['timestamp'],
                    df['realized_pnl'] + df['unrealized_pnl'],
                    label='Total')
            plt.title('P&L')
            plt.legend()
            plt.xticks(rotation=45)
            
            # Spread
            plt.subplot(2, 2, 4)
            plt.plot(df['timestamp'], df['spread'] * 100)
            plt.title('Spread (%)')
            plt.xticks(rotation=45)
            
            plt.tight_layout()
            plt.show()
        
        await asyncio.sleep(1)
    
    return pd.DataFrame(simulation_data)

# Run simulation
print("Running market making simulation for 60 seconds...")
simulation_df = await simulate_market_making()

print("\nSimulation Results:")
print(f"Realized P&L: ${simulation_df['realized_pnl'].iloc[-1]:,.2f}")
print(f"Total P&L: ${(simulation_df['realized_pnl'] + simulation_df['unrealized_pnl']).iloc[-1]:,.2f}")
print(f"Average Spread: {simulation_df['spread'].mean()*100:.4f}%")
print(f"Average Inventory: {simulation_df['inventory'].mean():.4f}")